# Un solo schema, sei modelli

**Classe:** quinta SIA · **Durata:** 2 ore

Con scikit-learn ogni modello si usa sempre nello stesso modo:

```python
modello = Algoritmo(parametri)        # 1. istanzia
modello.fit(X_train, y_train)         # 2. addestra
previsioni = modello.predict(X_test)  # 3. prevedi
valuta(y_test, previsioni)            # 4. valuta
```

Il codice è corto e uguale per tutti: la parte difficile sono le **scelte** sui dati e sulla valutazione.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
import os
import pandas as pd

# Se il notebook è aperto in Colab da GitHub, il CSV non è nella cartella del runtime:
# si legge direttamente dal repository. Chi copia il notebook in un altro repository
# cambia qui utente e nome del repository.
URL_BASE = "https://raw.githubusercontent.com/giovannipedroncelli/repo-demo-formatore/main/ml/dati/"

def leggi_csv(nome):
    percorso = os.path.join("dati", nome)
    if os.path.exists(percorso):
        return pd.read_csv(percorso)
    return pd.read_csv(URL_BASE + nome)

## 1. Regressione lineare: vendite e temperatura
Un chiosco registra le vendite di una bibita calda al variare della temperatura (dati del corso "Didattica dell'IA").

In [ ]:
temperature = np.array(range(0, 40, 2)).reshape(-1, 1)
vendite = [65, 58, 46, 45, 44, 42, 40, 40, 36, 38, 38, 28, 30, 22, 27, 25, 25, 20, 15, 5]
plt.plot(temperature, vendite, "o")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Vendite")
plt.show()

**Prevedi.** La retta avrà pendenza positiva o negativa? Circa quante vendite a 25 °C?

*La mia previsione:* …

In [ ]:
modello = LinearRegression()
modello.fit(temperature, vendite)
print("intercetta:", round(modello.intercept_, 2))
print("coefficiente:", np.round(modello.coef_, 3))
print("previsione a 25 °C:", round(modello.predict([[25]])[0], 1))

**Indaga.** Cosa significa, in parole, il coefficiente? Ha senso usare il modello per prevedere le vendite a 60 °C?

> Per la classe: prima di questa cella si può far scrivere la **discesa del gradiente da zero** in Python puro (notebook "Apprendimento supervisionato" del corso), e poi confrontare i due risultati.

## 2. Classificazione: quattro modelli, stesse righe

In [ ]:
funghi = leggi_csv("mushrooms.csv")
X = pd.get_dummies(funghi.drop(columns="class"))
y = (funghi["class"] == "p").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

**Prevedi.** Quale modello andrà meglio? Scrivi un'accuratezza per ciascuno.

*La mia previsione:* …

In [ ]:
modelli = {
    "baseline (classe più frequente)": DummyClassifier(strategy="most_frequent"),
    "regressione logistica": LogisticRegression(max_iter=1000),
    "kNN (k = 5)": KNeighborsClassifier(n_neighbors=5),
    "albero (profondità 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "SVM": SVC(),
}
for nome, m in modelli.items():
    m.fit(X_train, y_train)
    print(f"{nome:<32} {accuracy_score(y_test, m.predict(X_test)):.3f}")

**Indaga.** Tutti vicini al 100%: i modelli sono eccezionali oppure il problema è facile? Guarda cosa ha imparato l'albero.

In [ ]:
albero = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
print(export_text(albero, feature_names=list(X.columns)))

In [ ]:
solo_odore = pd.get_dummies(funghi[["odor"]])
a, b, c, d = train_test_split(solo_odore, y, test_size=0.2, stratify=y, random_state=42)
albero_odore = DecisionTreeClassifier(random_state=42).fit(a, c)
print("albero con la sola colonna odor:", round(accuracy_score(d, albero_odore.predict(b)), 3))

**Per la classe (livello 0):** stampare l'albero di profondità 2 e far classificare **a mano** tre funghi descritti a parole.

## 3. k-means: clienti con e senza scaling
Dati **inventati** di 100 clienti: reddito annuo ed età.

In [ ]:
rng = np.random.default_rng(0)
reddito = np.concatenate([rng.normal(20000, 3000, 50), rng.normal(60000, 5000, 50)])
eta = np.concatenate([rng.normal(25, 3, 25), rng.normal(55, 4, 25), rng.normal(25, 3, 25), rng.normal(55, 4, 25)])
clienti = pd.DataFrame({"reddito": reddito, "eta": eta})
clienti.describe().round(1)

**Prevedi.** Chiediamo 4 gruppi. Senza scaling, k-means troverà i quattro gruppi (reddito basso/alto × giovani/anziani)?

*La mia previsione:* …

In [ ]:
gruppi_senza = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(clienti)
gruppi_con = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(clienti))

fig, assi = plt.subplots(1, 2, figsize=(10, 4))
assi[0].scatter(clienti["reddito"], clienti["eta"], c=gruppi_senza)
assi[0].set_title("Senza scaling")
assi[1].scatter(clienti["reddito"], clienti["eta"], c=gruppi_con)
assi[1].set_title("Con StandardScaler")
for asse in assi:
    asse.set_xlabel("Reddito annuo (€)")
    asse.set_ylabel("Età")
plt.tight_layout()
plt.show()

**Indaga.** Perché senza scaling i gruppi seguono quasi solo il reddito? (Suggerimento: guarda di quanto variano i numeri delle due colonne.)

In [ ]:
inerzie = []
for k in range(1, 9):
    inerzie.append(KMeans(n_clusters=k, n_init=10, random_state=0).fit(StandardScaler().fit_transform(clienti)).inertia_)
plt.plot(range(1, 9), inerzie, "-o")
plt.xlabel("Numero di cluster k")
plt.ylabel("Inerzia")
plt.show()

### Modifica
Nel ciclo dei modelli, cambia `max_depth` dell'albero da 3 a 1: cosa succede all'accuratezza?

### Crea
Aggiungi al dizionario `modelli` un secondo kNN con `n_neighbors=1` e uno con `n_neighbors=50`. Cosa ti aspetti, e cosa succede?